# N4 — Model Context Protocol (MCP)

* Professor: Júlio Cesar dos Reis <a href="mailto:dosreis@unicamp.br">(dosreis@unicamp.br)</a>
* Monitor: Alejandro Núñez Arroyo <a href="mailto:r299215@dac.unicamp.br">(r299215@dac.unicamp.br)</a>
* Material base: Renan dos Santos Morais <a href="mailto:r299211@dac.unicamp.br">(r299211@dac.unicamp.br)</a>

Este notebook conecta uma aplicação a capacidades que vivem fora dela. O Model Context Protocol define um contrato de descoberta e uso dessas capacidades: um servidor declara ferramentas, dados e prompts em um formato fixo, e um cliente descobre o que existe e chama o que precisa, sem conhecer a implementação do outro lado.

O ponto de vista adotado é o de quem consome. Todas as células abrem uma conexão com um servidor que roda de verdade, em outro processo ou em outra máquina, e o que aparece na saída é a resposta do protocolo. As seções vão do handshake de uma conexão até um agente que escolhe ferramentas MCP com um modelo de linguagem e uma camada de política que filtra o que chega ao modelo.

## 0.1 Pré-requisitos

Este notebook pressupõe os seguintes conhecimentos:

- Python: funções, dicionários, anotações de tipo e gerenciadores de contexto;
- corrotinas com `async` e `await`, e execução de subprocessos;
- serialização em JSON e noção de JSON Schema;
- ferramentas ligadas a um modelo pelo esquema de argumentos, e o par pedido de chamada e observação;
- montagem de um agente com `create_agent` e leitura do histórico de mensagens.

## 0.2 Objetivos da aula

Objetivos deste notebook:

1. Distinguir os papéis de host, cliente e servidor, e a relação de um cliente por servidor.
2. Escrever um servidor MCP que declara ferramentas, um resource e um prompt.
3. Conectar um cliente pelo transporte stdio e ler a resposta do handshake.
4. Observar as mensagens JSON-RPC que cruzam o transporte durante uma sessão.
5. Descobrir as capacidades de um servidor pelos esquemas, sem consultar documentação.
6. Separar erro de execução, que chega com `isError`, de erro de protocolo, que chega como exceção.
7. Conectar o mesmo código de aplicação por três transportes: memória, stdio e HTTP.
8. Manter um cliente por servidor a partir de um arquivo de configuração `mcpServers`.
9. Converter as ferramentas de um servidor em ferramentas de LangChain e entregá-las a um agente.
10. Atender as requisições que o servidor faz ao cliente, `sampling` e `roots`.
11. Aplicar allowlist, timeout e aprovação humana no lado do cliente.

## 0.3 Mapa do notebook

Conteúdo deste notebook:

1. Papéis do protocolo.
2. Cliente stdio e handshake.
3. Mensagens do protocolo.
4. Descoberta das primitivas.
5. Erros vistos pelo cliente.
6. Transporte em memória.
7. Servidor remoto por HTTP.
8. Host com vários clientes.
9. Ferramentas MCP no agente.
10. Requisições do servidor ao cliente.
11. Política do cliente.
12. Exercícios de fixação.
13. Resumo da aula.
14. Referências.

## 0.4 Contexto usado nos exemplos

Os exemplos montam o assistente de um curso de agentes com LLMs. O curso tem três módulos: `A4`, sobre estado, nós e transições; `A5`, sobre grafos com chamadas de modelo; e `A6`, sobre ferramentas e agentes. Os alunos Julio, Renan e Ana têm percentual de conclusão registrado nesses módulos.

Esses dados ficam dentro de um servidor MCP escrito na seção 1, o `CursoMCPServer`, que roda como processo separado. Nenhuma célula lê o dicionário do servidor diretamente: o acesso é sempre pelo protocolo.

Três servidores aparecem no notebook, e cada um representa uma origem diferente:

| Servidor | Origem | Transporte |
|---|---|---|
| `curso` | escrito na seção 1 | stdio, subprocesso local |
| `filesystem` | servidor de referência publicado no npm | stdio, subprocesso local |
| `deepwiki` | serviço público na internet | Streamable HTTP |

## 0.5 Preparação do ambiente

As bibliotecas são `mcp` (SDK oficial do protocolo), `langchain` e `langgraph` (agente da seção 9), `langchain-ollama` e `langchain-google-genai` (clientes dos dois provedores de modelo) e `python-dotenv` (leitura das chaves). O pacote `mcp` cobre os dois lados do protocolo, e o notebook usa os dois: o lado do servidor na seção 1 e o lado do cliente nas demais.

A seção 8 usa o servidor de arquivos de referência, que roda por `npx` e exige Node.js instalado na máquina. A última célula do preâmbulo informa se o `npx` está disponível. Sem Node.js, as células da seção 8 relatam o servidor que ficou de fora e as demais seções continuam executando.

A seção 7 abre uma conexão com um serviço público na internet. As demais seções rodam sem rede depois da instalação, exceto as chamadas ao provedor de modelo das seções 9 e 10.

O acesso ao modelo hospedado exige uma chave de API. Os passos para obtê-la:

1. Criar uma conta em [ollama.com](https://ollama.com) e entrar nela.
2. Abrir [ollama.com/settings/keys](https://ollama.com/settings/keys) e gerar uma chave nova.
3. Copiar o valor exibido, que aparece uma única vez.
4. Guardar o valor na variável de ambiente `OLLAMA_API_KEY`.

Há três formas de cumprir o passo 4. Em uma máquina local, escreva a linha `OLLAMA_API_KEY=...` em um arquivo `.env` ao lado do notebook, que já está no `.gitignore`, e deixe o `load_dotenv()` da seção 0.6 carregá-la. A segunda forma é exportar a variável no ambiente antes de abrir o notebook. A terceira é a célula comentada abaixo, para o Google Colab. Não escreva a chave no código versionado.

In [ ]:
# Descomente se estiver em um ambiente sem as dependências instaladas.
%pip install -U "mcp>=1.29,<2" langchain langgraph langchain-ollama langchain-google-genai python-dotenv

A célula a seguir cobre o caso do Google Colab, onde não há arquivo `.env`. Descomente as três linhas, cole a chave entre as aspas e execute a célula. Torne a comentá-las antes de salvar o notebook, para que a chave não seja gravada no arquivo.

In [ ]:
# # Descomente no Google Colab, onde não há arquivo .env.
# import os
# os.environ["OLLAMA_API_KEY"] = ""
# os.environ["GOOGLE_API_KEY"] = ""

Agora importamos os elementos que serão usados ao longo da aula.

In [ ]:
import json
import os
import shutil
import sys
from contextlib import AsyncExitStack
from datetime import timedelta
from pathlib import Path

from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_core.messages import HumanMessage, ToolMessage
from langchain_core.tools import StructuredTool
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_ollama import ChatOllama

from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.client.streamable_http import streamablehttp_client
from mcp.shared.exceptions import McpError
from mcp.shared.memory import create_connected_server_and_client_session
from mcp.types import (
    CreateMessageRequestParams,
    CreateMessageResult,
    Implementation,
    ListRootsResult,
    Root,
    TextContent,
)

Glossário dos imports:

- `json`, `os`, `shutil`, `sys` e `Path`: formatação das respostas, leitura das chaves, verificação do `npx`, caminho do interpretador e arquivos do servidor;
- `AsyncExitStack`: mantém abertas as sessões de vários servidores ao mesmo tempo, na seção 8;
- `timedelta`: prazo de uma chamada de ferramenta, na seção 11;
- `load_dotenv`: leitura das chaves de API a partir do arquivo `.env`;
- `create_agent`: monta o grafo do agente com ferramentas, usado na seção 9;
- `HumanMessage` e `ToolMessage`: o pedido do aluno e o resultado de uma ferramenta no histórico do agente;
- `StructuredTool`: ferramenta de LangChain construída a partir de um esquema em JSON Schema, usada no adaptador da seção 9;
- `ChatOllama` e `ChatGoogleGenerativeAI`: clientes dos dois provedores configurados na seção 0.6;
- `ClientSession`: o cliente MCP, que fala o protocolo sobre um par de canais;
- `StdioServerParameters` e `stdio_client`: descrição e execução de um servidor local como subprocesso;
- `streamablehttp_client`: transporte para servidores remotos;
- `McpError`: exceção levantada quando a resposta traz o campo `error`;
- `create_connected_server_and_client_session`: liga cliente e servidor por canais em memória, sem processo;
- `CreateMessageRequestParams`, `CreateMessageResult`, `ListRootsResult`, `Root` e `TextContent`: tipos das requisições que o servidor faz ao cliente, na seção 10;
- `Implementation`: nome e versão que o cliente declara no handshake.

As três definições abaixo são usadas ao longo do notebook. `print_json` imprime estruturas de resposta; `PASTA` é o diretório de trabalho dos servidores locais, criado ao lado do notebook; e `LOG_SERVIDOR` recebe o que os servidores escrevem em `stderr`.

O desvio do `stderr` é um requisito de execução dentro de um notebook. O Jupyter substitui `sys.stderr` por um objeto sem descritor de arquivo, e o subprocesso do servidor precisa de um descritor real, sem o qual a conexão termina em `UnsupportedOperation: fileno`.

In [ ]:
def print_json(valor) -> None:
    """Imprime estruturas Python como JSON formatado."""
    print(json.dumps(valor, ensure_ascii=False, indent=2, default=str))


# O descritor precisa ser real: o stderr do kernel do Jupyter não tem fileno.
LOG_SERVIDOR = open(os.devnull, "w")

PASTA = Path("mcp_curso").resolve()
PASTA.mkdir(exist_ok=True)

(PASTA / "resumo.txt").write_text(
    "MCP: contrato de descoberta e uso de capacidades externas.\n", encoding="utf-8"
)

print("diretório de trabalho:", PASTA.name)
print("npx:", "disponível" if shutil.which("npx") else "ausente")

## 0.6 Escolhendo um provedor de LLM

As seções 9 e 10 chamam um modelo de linguagem. O notebook instancia o cliente do modelo por uma função única, `criar_modelo`, e a constante `PROVEDOR` seleciona qual dos dois é construído.

Os dois provedores diferem nos pontos da tabela abaixo, e `criar_modelo` concentra os que afetam a construção do cliente:

| Ponto | Ollama | Gemini |
|---|---|---|
| Classe | `ChatOllama` | `ChatGoogleGenerativeAI` |
| Conexão | `base_url` em `https://ollama.com` e cabeçalho `Authorization` entregue por `client_kwargs` | chave entregue em `api_key` |
| Limite de tokens da resposta | `num_predict` | `max_output_tokens` |
| Amostragem | `temperature` | fixa; a família Gemini 3.x não aceita `temperature`, `top_p` nem `top_k` |
| Raciocínio | `reasoning` | `thinking_level`, com os valores `minimal`, `low`, `medium` e `high` |
| Campo `content` da resposta | uma string | uma lista de blocos |

A seção 9 depende de tool calling, o mecanismo pelo qual o modelo devolve o nome de uma ferramenta declarada no pedido e os argumentos dela. O `gpt-oss:120b` e o `gemini-3.5-flash-lite` têm essa capacidade. Um modelo sem ela responde em texto e o agente encerra sem consultar o servidor MCP.

As saídas gravadas neste arquivo vêm do Ollama. Para executar pelo Gemini, gere uma chave em [aistudio.google.com/apikey](https://aistudio.google.com/apikey), guarde-a na variável `GOOGLE_API_KEY` e troque o valor de `PROVEDOR`.

O parâmetro `reasoning=False` retira o texto de raciocínio do `gpt-oss` do campo da resposta. Os tokens de raciocínio continuam sendo gerados e continuam contando no orçamento de `num_predict`. Quando o orçamento acaba antes da resposta visível, a chamada volta sem texto. O valor 1500 de `LIMITE_RESPOSTA` cobre as duas partes no percurso mais longo das seções 9 e 10.

In [ ]:
load_dotenv()

PROVEDOR = "ollama"  # troque para "gemini" depois de definir GOOGLE_API_KEY

MODELO_OLLAMA = "gpt-oss:120b"           # servido por https://ollama.com
MODELO_GEMINI = "gemini-3.5-flash-lite"  # servido pela API do Google AI Studio

URL_OLLAMA = "https://ollama.com"
TEMPERATURA = 0
LIMITE_RESPOSTA = 1500  # tokens gerados por chamada


def criar_modelo(provedor: str = PROVEDOR):
    """Instancia o cliente do provedor escolhido com o mesmo orçamento de geração."""
    if provedor == "ollama":
        chave = os.environ.get("OLLAMA_API_KEY")
        if not chave:
            raise RuntimeError(
                "OLLAMA_API_KEY não encontrada. Gere uma chave em "
                "https://ollama.com/settings/keys e defina a variável no arquivo .env, "
                "no ambiente do sistema ou na célula da seção 0.5."
            )

        return ChatOllama(
            model=MODELO_OLLAMA,
            base_url=URL_OLLAMA,
            client_kwargs={"headers": {"Authorization": f"Bearer {chave}"}},
            reasoning=False,
            temperature=TEMPERATURA,
            num_predict=LIMITE_RESPOSTA,
        )

    if provedor == "gemini":
        chave = os.environ.get("GOOGLE_API_KEY")
        if not chave:
            raise RuntimeError(
                "GOOGLE_API_KEY não encontrada. Gere uma chave em "
                "https://aistudio.google.com/apikey e defina a variável no arquivo .env "
                "ou no ambiente do sistema."
            )

        # A família Gemini 3.x não aceita temperature, top_p nem top_k.
        # O nível de raciocínio ocupa o lugar do reasoning do Ollama.
        return ChatGoogleGenerativeAI(
            model=MODELO_GEMINI,
            api_key=chave,
            max_output_tokens=LIMITE_RESPOSTA,
            thinking_level="low",
        )

    raise RuntimeError(f"Provedor não reconhecido: {provedor}. Use 'ollama' ou 'gemini'.")


modelo = criar_modelo()

print(PROVEDOR, "|", type(modelo).__name__)

---

# 1. Papéis do protocolo

O MCP organiza a integração em três papéis. O **host** é a aplicação que o usuário opera, e as decisões de uso das capacidades ficam nela. O **cliente** é o componente do host que fala o protocolo com um servidor. O **servidor** declara capacidades e as executa quando recebe um pedido.

A relação entre cliente e servidor é de um para um. Um host que usa três servidores mantém três clientes, cada um com sua sessão, sua numeração de mensagens e suas capacidades negociadas. A seção 8 monta esse arranjo.

O cliente responde por cinco tarefas, e nenhuma delas depende do domínio do servidor do outro lado:

| Tarefa | Operação executada |
|---|---|
| Transporte | executar um subprocesso ou abrir uma conexão HTTP |
| Handshake | enviar `initialize`, conferir a versão e enviar `notifications/initialized` |
| Correlação | gerar o `id` de cada requisição e casar cada resposta com o pedido correspondente |
| Descoberta | listar ferramentas, resources e prompts com os esquemas declarados |
| Tradução de erros | transformar o campo `error` da resposta em exceção da linguagem |

Três decisões ficam fora do cliente: qual ferramenta chamar, que é escolha do modelo ou da lógica do agente; o que fazer com o conteúdo devolvido, que é do host; e o controle do ciclo de decisão, que é do agente. O cliente entrega o catálogo e executa a chamada pedida, e a escolha entre as opções do catálogo acontece fora dele.

```text
usuário
  ↓
host
  ↓ mantém um cliente por servidor
clientes MCP
  ↓ trocam mensagens JSON-RPC pelo transporte configurado
servidores MCP
  ↓
dados, ferramentas, APIs, arquivos e prompts
```

A célula abaixo escreve o servidor do curso em um arquivo separado, porque o transporte stdio exige um processo. O `FastMCP` do SDK oficial declara cada capacidade por um decorador: `@mcp.tool()` para ferramentas, `@mcp.resource()` para dados endereçados por URI e `@mcp.prompt()` para mensagens parametrizadas.

In [ ]:
SERVIDOR = PASTA / "servidor_curso.py"

SERVIDOR.write_text('''"""Servidor MCP do curso: alvo dos clientes deste notebook."""

import json

from mcp.server.fastmcp import Context, FastMCP
from mcp.types import SamplingMessage, TextContent
from pydantic import BaseModel, Field

# log_level="ERROR" mantém os logs do servidor fora da saída das células.
mcp = FastMCP("CursoMCPServer", log_level="ERROR")

CRONOGRAMA = {
    "A4": {"titulo": "Estado, nos e transicoes", "duracao_min": 90, "pre_requisitos": []},
    "A5": {"titulo": "Grafos com chamadas de modelo", "duracao_min": 120, "pre_requisitos": ["A4"]},
    "A6": {"titulo": "Ferramentas e agentes", "duracao_min": 150, "pre_requisitos": ["A4", "A5"]},
}

PROGRESSO = {
    "julio": {"A4": 100, "A5": 100, "A6": 0},
    "renan": {"A4": 100, "A5": 50, "A6": 0},
    "ana": {"A4": 70, "A5": 0, "A6": 0},
}


class Modulo(BaseModel):
    codigo: str = Field(description="Codigo do modulo")
    titulo: str = Field(description="Titulo do modulo")
    duracao_min: int = Field(description="Duracao em minutos")
    pre_requisitos: list[str] = Field(description="Modulos exigidos antes deste")


class Plano(BaseModel):
    dias: int = Field(description="Dias necessarios")
    minutos_por_dia: int = Field(description="Minutos por dia")
    ultima_sessao_min: int = Field(description="Minutos da ultima sessao")


@mcp.tool()
def consultar_cronograma(codigo_modulo: str) -> Modulo:
    """Consulta titulo, duracao em minutos e pre-requisitos de um modulo do curso."""
    codigo = codigo_modulo.strip().upper()
    if codigo not in CRONOGRAMA:
        raise ValueError(f"Modulo {codigo} nao existe. Disponiveis: {', '.join(CRONOGRAMA)}.")
    return Modulo(codigo=codigo, **CRONOGRAMA[codigo])


@mcp.tool()
def consultar_progresso(nome_aluno: str) -> dict:
    """Consulta o percentual concluido por um aluno em cada modulo do curso."""
    chave = nome_aluno.strip().lower()
    if chave not in PROGRESSO:
        raise ValueError(f"Aluno {nome_aluno} nao registrado. Registrados: {', '.join(PROGRESSO)}.")
    return {"aluno": chave, "percentuais": PROGRESSO[chave]}


@mcp.tool()
def calcular_plano_estudo(carga_minutos: int, minutos_por_dia: int) -> Plano:
    """Calcula em quantos dias uma carga de estudo cabe na disponibilidade diaria."""
    if carga_minutos <= 0 or minutos_por_dia <= 0:
        raise ValueError("carga_minutos e minutos_por_dia devem ser positivos.")
    dias = -(-carga_minutos // minutos_por_dia)
    return Plano(
        dias=dias,
        minutos_por_dia=minutos_por_dia,
        ultima_sessao_min=carga_minutos - (dias - 1) * minutos_por_dia,
    )


@mcp.resource("curso://cronograma")
def cronograma() -> str:
    """Cronograma completo do curso, em JSON."""
    return json.dumps(CRONOGRAMA, ensure_ascii=False, indent=2)


@mcp.prompt()
def tutoria_modulo(codigo_modulo: str) -> str:
    """Prompt de tutoria sobre um modulo do curso."""
    codigo = codigo_modulo.strip().upper()
    if codigo not in CRONOGRAMA:
        raise ValueError(f"Modulo {codigo} nao existe.")
    modulo = CRONOGRAMA[codigo]
    return (
        "Atue como tutor de um curso de agentes com LLMs. "
        f"Explique o modulo {codigo}: {modulo['titulo']}, "
        f"com duracao de {modulo['duracao_min']} minutos."
    )


@mcp.tool()
async def resumir_modulo(codigo_modulo: str, ctx: Context) -> str:
    """Resume um modulo pedindo a geracao ao modelo do host (sampling)."""
    codigo = codigo_modulo.strip().upper()
    if codigo not in CRONOGRAMA:
        raise ValueError(f"Modulo {codigo} nao existe.")
    modulo = CRONOGRAMA[codigo]
    resultado = await ctx.session.create_message(
        messages=[
            SamplingMessage(
                role="user",
                content=TextContent(
                    type="text",
                    text=(
                        f"Resuma em uma frase o modulo {codigo} do curso: "
                        f"{modulo['titulo']}, com {modulo['duracao_min']} minutos."
                    ),
                ),
            )
        ],
        max_tokens=200,
    )
    return resultado.content.text


@mcp.tool()
async def pastas_autorizadas(ctx: Context) -> str:
    """Pergunta ao cliente quais pastas ele autoriza (roots)."""
    roots = await ctx.session.list_roots()
    if not roots.roots:
        return "O cliente nao declarou nenhuma raiz."
    return "Raizes autorizadas: " + ", ".join(raiz.name or "sem nome" for raiz in roots.roots)


if __name__ == "__main__":
    mcp.run()
''', encoding="utf-8")

print("servidor escrito em:", SERVIDOR.parent.name + "/" + SERVIDOR.name)

## 1.1 Discussão

O arquivo declara sete capacidades de três famílias. Cinco são ferramentas, uma é resource e uma é prompt. A separação entre elas está no efeito esperado: um resource devolve conteúdo endereçado por URI, um prompt devolve mensagens prontas para uma conversa, e uma ferramenta executa código com os argumentos recebidos.

Os esquemas que o cliente vai receber saem do próprio código Python. As anotações de tipo dos parâmetros viram o `inputSchema` em JSON Schema, a docstring vira a descrição, e a anotação de retorno vira o `outputSchema`. Os três casos possíveis aparecem no arquivo: um modelo Pydantic, como o de `consultar_cronograma`, vira o esquema do objeto inteiro; um `-> str`, como o de `resumir_modulo`, vira um objeto de um campo só, chamado `result`; e o `-> dict` de `consultar_progresso`, sem parâmetros de tipo, não produz esquema nenhum.

As duas últimas ferramentas recebem um parâmetro `ctx` e o usam para fazer requisições ao cliente. Elas dependem de o cliente aceitar essas requisições, e a seção 10 registra os handlers que faltam para que executem.

O texto do servidor está sem acentos porque ele é gravado dentro de uma string do notebook, e essa é a única diferença entre este arquivo e um servidor escrito à parte. O arquivo gravado em `mcp_curso/servidor_curso.py` roda fora do notebook por `python servidor_curso.py`, como qualquer servidor MCP.

---

# 2. Cliente stdio e handshake

O transporte **stdio** executa o servidor como subprocesso e conversa por `stdin` e `stdout`. A conexão tem três camadas, e cada uma é um objeto diferente:

1. `StdioServerParameters` descreve como iniciar o processo: o comando, os argumentos e o ambiente;
2. `stdio_client` executa o processo e devolve os dois canais, de leitura e de escrita;
3. `ClientSession` é o cliente MCP, que fala o protocolo sobre esses canais.

O método `initialize` executa o handshake inteiro: envia a requisição com a versão de protocolo e as capacidades do cliente, recebe as capacidades do servidor e envia a notificação `notifications/initialized`. O que ele devolve é a resposta do servidor.

In [ ]:
PARAMS_CURSO = StdioServerParameters(
    command=sys.executable,
    args=[str(SERVIDOR)],
    env=dict(os.environ),
)

CLIENTE = Implementation(name="TutorLangGraphHost", version="1.0.0")


async def apresentar_servidor() -> None:
    """Abre uma sessão com o servidor do curso e imprime a resposta do handshake."""
    async with stdio_client(PARAMS_CURSO, errlog=LOG_SERVIDOR) as (leitura, escrita):
        async with ClientSession(leitura, escrita, client_info=CLIENTE) as sessao:
            inicio = await sessao.initialize()

            print("servidor:", inicio.serverInfo.name, inicio.serverInfo.version)
            print("versão de protocolo negociada:", inicio.protocolVersion)
            print("capacidades declaradas pelo servidor:")
            print_json(inicio.capabilities.model_dump(exclude_none=True))


await apresentar_servidor()

## 2.1 Leitura da execução

O servidor devolveu três famílias de capacidade: `tools`, `resources` e `prompts`. Essa resposta autoriza o cliente a chamar cada família de métodos. Um servidor que declarasse apenas `tools` recusaria `resources/list` com o código `-32601`, de método não encontrado.

O campo `listChanged` indica se o servidor emite uma notificação quando a lista muda. Com o valor `False`, o cliente que quiser detectar uma ferramenta nova precisa listar de novo. A versão impressa ao lado do nome é a do SDK que construiu o servidor, porque o arquivo da seção 1 não declara versão própria.

A sessão vive dentro do `async with`. Ao sair do bloco, os canais são fechados e o subprocesso termina. Um host de aplicação mantém esse bloco aberto enquanto a aplicação roda, e é por isso que as células deste notebook abrem e fecham uma sessão por exemplo: cada célula precisa rodar sozinha.

---

# 3. Mensagens do protocolo

O MCP transporta mensagens JSON-RPC 2.0. São três formas. Uma **requisição** tem `id` e `method`, e pede uma operação ao outro lado. Uma **resposta** repete o `id` da requisição e traz `result` ou `error`. Uma **notificação** tem `method` e não tem `id`, porque nenhuma resposta é esperada.

Essas mensagens ficam escondidas atrás dos métodos de `ClientSession`. Para observá-las, a célula abaixo coloca um objeto entre a sessão e cada canal do transporte. Ele repassa cada mensagem e guarda uma cópia, sem alterar o que trafega.

Fluxo da sessão observada:

```text
initialize          → requisição com id, resposta com o mesmo id
notifications/initialized → notificação, sem id e sem resposta
tools/list          → requisição com id, resposta com o catálogo
tools/call          → requisição com nome e argumentos, resposta com o resultado
```

In [ ]:
TRAFEGO: list[tuple[str, dict]] = []


class CanalObservado:
    """Repassa as mensagens de um canal do transporte e guarda uma cópia de cada uma."""

    def __init__(self, canal, sentido: str) -> None:
        self.canal = canal
        self.sentido = sentido

    def registrar(self, item) -> None:
        TRAFEGO.append((self.sentido, item.message.model_dump(by_alias=True, exclude_none=True)))

    async def send(self, item) -> None:
        self.registrar(item)
        await self.canal.send(item)

    def __aiter__(self):
        return self

    async def __anext__(self):
        item = await self.canal.__anext__()
        self.registrar(item)
        return item

    async def __aenter__(self):
        await self.canal.__aenter__()
        return self

    async def __aexit__(self, *excecao):
        return await self.canal.__aexit__(*excecao)


async def observar_sessao() -> None:
    """Executa quatro operações com os canais observados."""
    async with stdio_client(PARAMS_CURSO, errlog=LOG_SERVIDOR) as (leitura, escrita):
        async with ClientSession(
            CanalObservado(leitura, "servidor -> cliente"),
            CanalObservado(escrita, "cliente -> servidor"),
            client_info=CLIENTE,
        ) as sessao:
            await sessao.initialize()
            await sessao.list_tools()
            await sessao.call_tool(
                "calcular_plano_estudo", {"carga_minutos": 150, "minutos_por_dia": 45}
            )


await observar_sessao()

for sentido, mensagem in TRAFEGO:
    print(f"{sentido}: {json.dumps(mensagem, ensure_ascii=False)[:150]}")

print("\núltima mensagem, sem truncar:")
print_json(TRAFEGO[-1][1])

## 3.1 Leitura da execução

A sequência gravada acima tem sete mensagens:

1. `initialize` sai do cliente com `id` 0, a versão de protocolo pretendida e o `clientInfo` declarado em `CLIENTE`;
2. a resposta volta com o mesmo `id` 0, as capacidades do servidor e o `serverInfo`;
3. `notifications/initialized` sai do cliente sem `id`, e nada volta;
4. `tools/list` sai com `id` 1, sem parâmetros;
5. a resposta com `id` 1 traz o catálogo inteiro, com os esquemas de cada ferramenta;
6. `tools/call` sai com `id` 2, e os parâmetros trazem `name` e `arguments`;
7. a resposta com `id` 2 traz `content` e `structuredContent`, impressa sem truncar no fim da célula.

O `id` liga uma resposta ao pedido que a originou. Ele não descreve ordem de chegada: várias requisições podem estar pendentes ao mesmo tempo no mesmo transporte, e a correlação por `id` sustenta esse caso.

O conteúdo dessas mensagens é o mesmo em qualquer transporte. O que muda entre stdio e HTTP é como os bytes chegam do outro lado, e as seções 6 e 7 trocam o transporte sem alterar nenhuma chamada de `ClientSession`.

---

# 4. Descoberta das primitivas

Um cliente usa um servidor que nunca viu a partir dos metadados que ele declara. Para ferramentas, os metadados são o nome, a descrição, o `inputSchema` e, quando existe, o `outputSchema`. São esses campos que o host repassa ao modelo na seção 9.

A função `com_sessao_curso` abaixo concentra a abertura da sessão, e as células seguintes passam para ela apenas o que fazer com a sessão aberta. Os parâmetros extras que ela repassa a `ClientSession` são usados na seção 10.

In [ ]:
async def com_sessao_curso(funcao, **parametros):
    """Abre uma sessão com o servidor do curso e entrega a sessão à `funcao`."""
    async with stdio_client(PARAMS_CURSO, errlog=LOG_SERVIDOR) as (leitura, escrita):
        async with ClientSession(leitura, escrita, client_info=CLIENTE, **parametros) as sessao:
            await sessao.initialize()
            return await funcao(sessao)


async def descobrir(sessao: ClientSession) -> None:
    """Lista as três famílias de capacidade com os metadados declarados."""
    catalogo = await sessao.list_tools()
    print(f"FERRAMENTAS ({len(catalogo.tools)}):")
    for ferramenta in catalogo.tools:
        obrigatorios = ", ".join(ferramenta.inputSchema.get("required", []))
        saida = "tipada" if ferramenta.outputSchema else "texto"
        print(f"  {ferramenta.name}({obrigatorios}) -> saída {saida}")
        print(f"      {ferramenta.description}")

    recursos = await sessao.list_resources()
    print(f"\nRESOURCES ({len(recursos.resources)}):")
    for recurso in recursos.resources:
        print(f"  {recurso.uri}  ({recurso.mimeType})")

    prompts = await sessao.list_prompts()
    print(f"\nPROMPTS ({len(prompts.prompts)}):")
    for prompt in prompts.prompts:
        argumentos = ", ".join(a.name for a in (prompt.arguments or []))
        print(f"  {prompt.name}({argumentos})")


await com_sessao_curso(descobrir)

## 4.1 Chamada das três famílias

Cada família tem um método próprio e um formato de resposta próprio. A célula abaixo chama as três e imprime o campo que carrega o conteúdo em cada caso.

In [ ]:
async def usar_primitivas(sessao: ClientSession) -> None:
    """Chama uma ferramenta, lê um resource e obtém um prompt parametrizado."""
    resultado = await sessao.call_tool("consultar_cronograma", {"codigo_modulo": "a6"})
    print("FERRAMENTA -> isError:", resultado.isError)
    print("  content[0].text:  ", resultado.content[0].text.replace("\n", " "))
    print("  structuredContent:", resultado.structuredContent)

    conteudo = await sessao.read_resource("curso://cronograma")
    print("\nRESOURCE ->", conteudo.contents[0].mimeType,
          f"({len(conteudo.contents[0].text)} caracteres)")

    prompt = await sessao.get_prompt("tutoria_modulo", {"codigo_modulo": "A6"})
    mensagem = prompt.messages[0]
    print("\nPROMPT -> role:", mensagem.role)
    print("  ", mensagem.content.text)


await com_sessao_curso(usar_primitivas)

## 4.2 Leitura da execução

A ferramenta devolveu dois campos com o mesmo dado. O `content` é a lista de blocos que o modelo lê como texto, e o `structuredContent` é o objeto validado contra o `outputSchema`, que a aplicação lê como dado. A ferramenta `consultar_progresso`, sem esquema de saída, devolve apenas `content`.

A coluna impressa na listagem anterior vem da presença desse esquema, e não do tipo devolvido pelo servidor. `resumir_modulo` e `pastas_autorizadas` devolvem `str` e aparecem como tipadas, porque a anotação de retorno produziu um esquema de um campo. `consultar_progresso` devolve um dicionário e aparece como texto, porque a anotação `dict` sem parâmetros de tipo não descreve as chaves.

O argumento enviado foi `"a6"`, em minúsculas, e o servidor normalizou para `A6`. A normalização é código do servidor, e não do protocolo: o `inputSchema` declara o tipo `string` e nada além disso. Validação de formato e de faixa fica dentro da ferramenta.

O resource devolveu `contents`, uma lista com URI, `mimeType` e texto. O prompt devolveu `messages`, com papel e conteúdo, no formato que uma conversa com o modelo aceita. Os três formatos são fixos e independem do domínio do servidor, condição de que um cliente genérico depende.

---

# 5. Erros vistos pelo cliente

Duas coisas diferentes chegam ao cliente como erro, e o host trata cada uma de um jeito. O **erro de execução** acontece dentro da ferramenta: a resposta é um resultado normal, com o campo `isError` verdadeiro e o texto da falha em `content`. Nenhuma exceção é levantada, e o texto entra no histórico do modelo como qualquer outra observação.

O **erro de protocolo** acontece antes ou fora da execução: método inexistente, parâmetro inválido, recurso não encontrado. A resposta traz o campo `error`, e o SDK a converte em `McpError`.

A célula abaixo provoca três casos. Os dois primeiros são chamadas de ferramenta, uma com argumento recusado pela ferramenta e outra com nome que não existe no servidor; o terceiro é a leitura de um resource inexistente.

In [ ]:
async def demonstrar_erros(sessao: ClientSession) -> None:
    """Provoca um erro de execução, uma ferramenta inexistente e um erro de protocolo."""
    falha = await sessao.call_tool("consultar_cronograma", {"codigo_modulo": "A9"})
    print("1) módulo inexistente -> isError:", falha.isError)
    print("   ", falha.content[0].text)

    inexistente = await sessao.call_tool("nao_existe", {})
    print("\n2) ferramenta inexistente -> isError:", inexistente.isError)
    print("   ", inexistente.content[0].text)

    try:
        await sessao.read_resource("curso://inexistente")
    except McpError as erro:
        print("\n3) resource inexistente -> exceção McpError")
        print("    code:", erro.error.code, "| message:", erro.error.message)


await com_sessao_curso(demonstrar_erros)

## 5.1 Discussão

O primeiro caso é o previsto: a ferramenta levantou `ValueError`, o servidor embrulhou a mensagem em um resultado com `isError` verdadeiro, e o texto traz a lista de módulos aceitos. O modelo que receber essa observação tem a informação necessária para refazer a chamada.

O segundo caso diverge da especificação, que classifica uma ferramenta desconhecida como erro de protocolo `-32602`. Este servidor devolveu um resultado com `isError` verdadeiro. O terceiro caso diverge no código: a especificação reserva `-32002` para recurso não encontrado, e o código devolvido foi outro.

A consequência para quem escreve o cliente é que o código numérico não serve como critério de fluxo. A distinção que se mantém é estrutural, e vale nos dois casos:

- resultado com `isError` verdadeiro: o texto volta ao modelo, que pode corrigir os argumentos e tentar de novo;
- exceção: a integração falhou, o registro vai para o log da aplicação e a chamada não é repetida com os mesmos dados.

---

# 6. Transporte em memória

Um servidor que roda no mesmo programa do cliente dispensa o processo. O SDK liga os dois por canais em memória, e o protocolo inteiro continua acontecendo: handshake, serialização das mensagens e correlação por `id`. A célula abaixo importa o módulo do servidor escrito na seção 1 e conecta o cliente ao objeto `mcp` dele, sem executar `mcp.run()`.

In [ ]:
sys.path.insert(0, str(PASTA))

from servidor_curso import mcp as objeto_servidor  # noqa: E402


async def sessao_em_memoria() -> None:
    """Conecta cliente e servidor por canais em memória, sem subprocesso."""
    async with create_connected_server_and_client_session(objeto_servidor) as sessao:
        inicio = await sessao.initialize()
        print("servidor:", inicio.serverInfo.name, "| transporte: memória")

        resultado = await sessao.call_tool("consultar_progresso", {"nome_aluno": "Renan"})
        print("resultado:", resultado.content[0].text.replace("\n", " "))


await sessao_em_memoria()

## 6.1 Discussão

As chamadas são as mesmas da seção 2, `initialize` e `call_tool`, com o mesmo formato de resposta. O que saiu foi a execução do processo e o par de canais do sistema operacional, substituídos por filas dentro do mesmo interpretador. Esse é o transporte dos testes automatizados de um servidor próprio: sem processo e sem porta, a suíte não depende de o servidor subir, e ainda assim percorre o caminho do protocolo, incluindo a validação dos argumentos contra o `inputSchema` e a serialização do resultado.

O import do módulo torna esse transporte possível e limita seu alcance: ele exige que o código do servidor esteja instalado no mesmo interpretador. Um servidor de terceiros escrito em outra linguagem só é alcançável por stdio ou HTTP.

---

# 7. Servidor remoto por HTTP

O transporte **Streamable HTTP** conecta o cliente a um servidor que roda em outra máquina. O endereço substitui o comando, e nenhum processo é executado localmente.

O servidor usado aqui é o DeepWiki, um serviço público que responde perguntas sobre repositórios do GitHub e não exige chave. A comparação com a seção 2 é direta: muda a função de transporte, e `ClientSession` e as chamadas continuam iguais.

In [ ]:
URL_REMOTA = "https://mcp.deepwiki.com/mcp"


async def consultar_servidor_remoto(repositorio: str) -> None:
    """Conecta a um servidor remoto por Streamable HTTP e chama uma ferramenta dele."""
    async with streamablehttp_client(URL_REMOTA) as (leitura, escrita, _):
        async with ClientSession(leitura, escrita, client_info=CLIENTE) as sessao:
            inicio = await sessao.initialize()
            print("servidor remoto:", inicio.serverInfo.name,
                  "| protocolo:", inicio.protocolVersion)

            catalogo = await sessao.list_tools()
            print("ferramentas:", [f.name for f in catalogo.tools], "\n")

            resultado = await sessao.call_tool(
                "read_wiki_structure", {"repoName": repositorio}
            )
            print(resultado.content[0].text[:300], "...")


await consultar_servidor_remoto("modelcontextprotocol/python-sdk")

## 7.1 Discussão

O contrato não muda com o transporte. O mesmo `list_tools` e o mesmo `call_tool` serviram para um subprocesso local, para um objeto em memória e agora para um serviço na internet. Um host que troque a entrada de configuração de `command` para `url` não altera o resto do código.

O que muda são as condições de operação. Um servidor remoto acrescenta latência, indisponibilidade e limite de requisições ao conjunto de falhas possíveis, o que torna o prazo da seção 11 uma exigência e não uma escolha. Servidores remotos privados acrescentam ainda a autenticação, em geral por OAuth, com um fluxo que depende de credenciais e de um navegador.

---

# 8. Host com vários clientes

Um host lê um arquivo de configuração e mantém um cliente por entrada. O formato `mcpServers` é o mesmo usado por aplicações como Claude Desktop, Cursor e VS Code, e tem duas famílias de entrada: `command` mais `args` para um servidor local, e `url` para um remoto.

Duas decisões de segurança ficam nesse arquivo. Segredos vão em `env`, que é o ambiente do subprocesso, e nunca em `args`, que aparece na linha de comando visível na lista de processos. E `args` delimita privilégio: o servidor de arquivos enxerga apenas os diretórios que recebe como argumento, e trocar essa entrada por `/` daria a ele o disco inteiro.

Uma entrada com chave de API tem esta forma, e o valor é lido do ambiente do host antes de chegar ao arquivo:

```json
"github": {
  "command": "python3",
  "args": ["servidor_github.py"],
  "env": { "GITHUB_PERSONAL_ACCESS_TOKEN": "${GITHUB_TOKEN}" }
}
```

A configuração deste notebook declara os três servidores da seção 0.4. Os caminhos são montados em tempo de execução, e a impressão mostra a chave que define o transporte de cada entrada, sem gravar o caminho da máquina no arquivo.

In [ ]:
CONFIG_MCP = {
    "mcpServers": {
        # Servidor escrito na seção 1, executado pelo interpretador deste notebook.
        "curso": {"command": sys.executable, "args": [str(SERVIDOR)]},
        # Servidor de terceiros publicado no npm. O caminho em `args` é tudo o que ele enxerga.
        "filesystem": {
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", str(PASTA)],
        },
        # Servidor remoto: sem processo local, só o endereço.
        "deepwiki": {"url": URL_REMOTA},
    }
}

for nome, entrada in CONFIG_MCP["mcpServers"].items():
    if "url" in entrada:
        print(f"{nome:11} url      -> {entrada['url']}  (Streamable HTTP)")
    else:
        print(f"{nome:11} command  -> {Path(entrada['command']).name}  (stdio, subprocesso)")

## 8.1 Abertura de uma sessão por entrada

A configuração não executa nada por si. O host percorre as entradas, abre uma sessão pelo transporte que cada entrada indica e guarda todas em uma pilha de contextos. `AsyncExitStack` mantém os blocos `async with` abertos além do corpo de uma função e os fecha na ordem inversa ao sair. A última chamada da célula usa uma ferramenta do servidor de arquivos para ler o `resumo.txt` criado no preâmbulo.

In [ ]:
async def abrir_sessao(pilha: AsyncExitStack, entrada: dict) -> ClientSession:
    """Abre uma sessão para uma entrada de configuração, pelo transporte que ela indica."""
    if "url" in entrada:
        leitura, escrita, _ = await pilha.enter_async_context(
            streamablehttp_client(entrada["url"])
        )
    else:
        parametros = StdioServerParameters(
            command=entrada["command"],
            args=entrada.get("args", []),
            env={**os.environ, **entrada.get("env", {})},
        )
        leitura, escrita = await pilha.enter_async_context(
            stdio_client(parametros, errlog=LOG_SERVIDOR)
        )

    sessao = await pilha.enter_async_context(
        ClientSession(leitura, escrita, client_info=CLIENTE)
    )
    await sessao.initialize()
    return sessao


async def catalogo_do_host() -> None:
    """Conecta a todos os servidores da configuração e imprime o catálogo agregado."""
    async with AsyncExitStack() as pilha:
        sessoes: dict[str, ClientSession] = {}

        for nome, entrada in CONFIG_MCP["mcpServers"].items():
            try:
                sessoes[nome] = await abrir_sessao(pilha, entrada)
            except Exception as erro:
                print(f"{nome}: não conectou ({type(erro).__name__}), ficará de fora")

        for nome, sessao in sessoes.items():
            catalogo = await sessao.list_tools()
            nomes = [f.name for f in catalogo.tools]
            print(f"{nome:11} {len(nomes):2} ferramentas   ex.: {nomes[:2]}")

        if "filesystem" in sessoes:
            arquivo = await sessoes["filesystem"].call_tool(
                "read_text_file", {"path": str(PASTA / "resumo.txt")}
            )
            print("\nfilesystem -> resumo.txt:", arquivo.content[0].text.strip())


await catalogo_do_host()

## 8.2 Leitura da execução

Três servidores, três transportes diferentes e um catálogo agregado. O host conhece a origem de cada ferramenta porque guardou a sessão sob o nome da entrada, e o roteamento de uma chamada é a escolha da sessão no dicionário.

O `try` na abertura muda o comportamento em caso de falha. Sem ele, um servidor que não sobe interrompe o laço e derruba o host inteiro. Com ele, a falha é relatada e os demais servidores continuam disponíveis, que é o comportamento esperado quando parte das entradas é de terceiros.

Nomes colidem entre servidores. O servidor de arquivos expõe `read_text_file`, e outro servidor poderia expor uma ferramenta com o mesmo nome. Quando as ferramentas dos vários servidores são reunidas em uma lista única, como acontece na seção 9, o nome precisa receber o prefixo do servidor de origem.

---

# 9. Ferramentas MCP no agente

Um agente de LangChain recebe ferramentas com nome, descrição e esquema de argumentos. Uma ferramenta MCP declara exatamente esses três campos. A conversão entre os dois formatos é o adaptador da célula abaixo: ele lê o catálogo de uma sessão e devolve um `StructuredTool` por ferramenta, cuja execução é uma chamada `call_tool` na mesma sessão.

Fluxo de uma pergunta:

```text
pergunta do aluno
  ↓
modelo com as ferramentas MCP ligadas
  ↓
AIMessage com tool_calls           ← o modelo escolhe nome e argumentos
  ↓
StructuredTool chama session.call_tool
  ↓
servidor MCP executa em outro processo
  ↓
ToolMessage com o resultado        ← observação
  ↓
AIMessage sem tool_calls           ← resposta final
```

A sessão precisa estar aberta enquanto o agente roda, porque cada ferramenta a usa na execução. Por isso o `create_agent` e o `ainvoke` ficam dentro do bloco da sessão.

In [ ]:
FERRAMENTAS_DO_AGENTE = {"consultar_cronograma", "consultar_progresso", "calcular_plano_estudo"}

PROMPT_ASSISTENTE = """Atue como assistente do curso de agentes com LLMs.
O curso tem os módulos A4, A5 e A6.
Consulte as ferramentas antes de afirmar duração, pré-requisito ou progresso.
Responda em português, em no máximo quatro frases."""


def adaptar_ferramenta(sessao: ClientSession, ferramenta) -> StructuredTool:
    """Converte uma ferramenta MCP em ferramenta de LangChain ligada à sessão."""

    async def executar(**argumentos) -> str:
        resultado = await sessao.call_tool(ferramenta.name, argumentos)
        return resultado.content[0].text

    return StructuredTool(
        name=ferramenta.name,
        description=ferramenta.description or "",
        args_schema=ferramenta.inputSchema,
        coroutine=executar,
    )


async def responder(pergunta: str) -> None:
    """Monta um agente com as ferramentas do servidor do curso e responde à pergunta."""
    async with stdio_client(PARAMS_CURSO, errlog=LOG_SERVIDOR) as (leitura, escrita):
        async with ClientSession(leitura, escrita, client_info=CLIENTE) as sessao:
            await sessao.initialize()

            catalogo = await sessao.list_tools()
            ferramentas = [
                adaptar_ferramenta(sessao, f)
                for f in catalogo.tools
                if f.name in FERRAMENTAS_DO_AGENTE
            ]

            agente = create_agent(
                model=modelo, tools=ferramentas, system_prompt=PROMPT_ASSISTENTE
            )
            resultado = await agente.ainvoke({"messages": [HumanMessage(pergunta)]})

    for mensagem in resultado["messages"]:
        pedidos = [chamada["name"] for chamada in getattr(mensagem, "tool_calls", [])]
        rotulo = "resultado" if isinstance(mensagem, ToolMessage) else "texto"
        conteudo = pedidos or f"{rotulo}: {mensagem.text[:110]}"
        print(f"{type(mensagem).__name__:13} | {conteudo}")


await responder(
    "Sou o aluno Renan. Quanto dura o módulo A6 e em quantos dias eu o concluo "
    "estudando 45 minutos por dia?"
)

## 9.1 Leitura da execução

O histórico impresso registra o percurso da pergunta. O modelo pediu o cronograma do A6, recebeu a duração de 150 minutos, pediu `calcular_plano_estudo` com esse valor e produziu o texto final. O argumento do segundo pedido veio do resultado do primeiro, e nenhum dos dois valores estava no prompt.

O adaptador tem três linhas de contrato. O `name` e o `description` vêm do servidor e são o que o modelo compara com a pergunta. O `args_schema` recebe o `inputSchema` em JSON Schema, sem conversão para classe Pydantic. E a execução devolve o texto do primeiro bloco de `content`, que é o formato que uma `ToolMessage` aceita.

O filtro por `FERRAMENTAS_DO_AGENTE` deixou de fora as duas ferramentas que dependem de handlers do cliente. Esse filtro antecipa a allowlist da seção 11: o catálogo do servidor tem sete capacidades, e três chegaram ao modelo.

O restante da sequência depende do provedor e do modelo. A pergunta identifica o aluno, e o modelo pode consultar o progresso dele também, na mesma resposta que pede o cronograma ou em uma resposta seguinte. Só o par cronograma e plano de estudo é fixo, porque o argumento do segundo depende do resultado do primeiro.

---

# 10. Requisições do servidor ao cliente

O protocolo é bidirecional. Depois do handshake, o servidor pode enviar requisições ao cliente, e são três as capacidades que o cliente declara para recebê-las:

| Capacidade | O servidor pede | Efeito |
|---|---|---|
| `sampling` | uma geração ao modelo do host | o servidor usa um modelo sem ter chave nem escolher o provedor |
| `roots` | a lista de diretórios que ele pode acessar | o cliente delimita o escopo do servidor |
| `elicitation` | um dado ao usuário, durante a execução | confirmação ou informação que faltou |

As duas últimas ferramentas do servidor do curso dependem disso. `resumir_modulo` envia `sampling/createMessage` e espera um texto gerado; `pastas_autorizadas` envia `roots/list` e espera a lista de diretórios.

O cliente declara essas capacidades registrando os handlers. Sem eles, as duas ferramentas falham, e é o registro que autoriza o servidor a pedir. O handler de sampling abaixo chama o modelo da seção 0.6, e é nesse ponto que uma aplicação pediria a aprovação do usuário antes de gastar tokens.

In [ ]:
async def tratar_sampling(contexto, parametros: CreateMessageRequestParams) -> CreateMessageResult:
    """Atende um pedido de geração do servidor com o modelo do host."""
    pedido = parametros.messages[0].content.text
    print("  [host] geração pedida pelo servidor:", pedido[:70], "...")

    resposta = await modelo.ainvoke([HumanMessage(pedido)])

    return CreateMessageResult(
        role="assistant",
        content=TextContent(type="text", text=resposta.text),
        model=MODELO_OLLAMA if PROVEDOR == "ollama" else MODELO_GEMINI,
    )


async def tratar_roots(contexto) -> ListRootsResult:
    """Informa ao servidor quais diretórios o cliente autoriza."""
    print("  [host] lista de diretórios pedida pelo servidor")
    return ListRootsResult(roots=[Root(uri=PASTA.as_uri(), name="material do curso")])


async def usar_handlers(sessao: ClientSession) -> None:
    """Chama as duas ferramentas que dependem de requisições ao cliente."""
    resumo = await sessao.call_tool("resumir_modulo", {"codigo_modulo": "A6"})
    print("  resumo:", resumo.content[0].text[:160], "\n")

    raizes = await sessao.call_tool("pastas_autorizadas", {})
    print("  raízes:", raizes.content[0].text)


await com_sessao_curso(
    usar_handlers,
    sampling_callback=tratar_sampling,
    list_roots_callback=tratar_roots,
)

## 10.1 Discussão

As linhas marcadas com `[host]` foram impressas por código do cliente, executado a pedido do servidor. A ordem da execução inverte a da seção 2: a requisição partiu do servidor, e a resposta foi produzida pelo host.

O argumento de governança do `sampling` está no que o servidor não recebeu. Ele obteve um texto gerado sem ver a chave de API, sem escolher o modelo e sem saber o custo da chamada. Trocar o valor de `PROVEDOR` na seção 0.6 muda o modelo que responde, e o código do servidor permanece igual.

A especificação recomenda aprovação humana nesse ponto. O cliente pode recusar a requisição, e a recusa é uma resposta válida do protocolo. O `tratar_roots` cumpre a mesma função para o sistema de arquivos: o servidor pergunta onde pode trabalhar, e a resposta é escrita pelo cliente, não pelo servidor.

---

# 11. Política do cliente

Um host que consome servidores de terceiros executa código que não escreveu, com descrições que não revisou. Quatro controles ficam do lado do cliente, antes de a requisição existir.

**Allowlist.** O catálogo do servidor não precisa chegar inteiro ao modelo. O servidor de arquivos declara catorze ferramentas, incluindo escrita e movimentação. Uma aplicação que apenas lê expõe apenas a leitura.

**Prazo.** `call_tool` aceita `read_timeout_seconds`. Sem ele, um servidor que não responde deixa o agente parado.

**Aprovação humana.** Ferramentas com efeito destrutivo ou irreversível passam por confirmação, e o mesmo vale para os pedidos de `sampling` da seção 10.

**Desconfiança do conteúdo.** Descrições de ferramentas e resultados de servidores de terceiros entram no contexto do modelo. Um servidor pode escrever instruções nesse texto, e o canal de entrega é o mesmo dos dados. O texto vindo do servidor é dado, e não instrução.

In [ ]:
FERRAMENTAS_PERMITIDAS = {
    "consultar_cronograma",
    "consultar_progresso",
    "calcular_plano_estudo",
    "resumir_modulo",
}

FERRAMENTAS_SENSIVEIS = {"resumir_modulo"}  # consome tokens do modelo do host

PRAZO = timedelta(seconds=15)


async def chamar_com_politica(
    sessao: ClientSession,
    nome: str,
    argumentos: dict,
    aprovado_pelo_usuario: bool = False,
) -> dict:
    """Aplica allowlist, aprovação e prazo antes de enviar a requisição ao servidor."""
    if nome not in FERRAMENTAS_PERMITIDAS:
        return {"enviado": False, "motivo": "fora_da_allowlist", "ferramenta": nome}

    if nome in FERRAMENTAS_SENSIVEIS and not aprovado_pelo_usuario:
        return {"enviado": False, "motivo": "requer_aprovacao_humana", "ferramenta": nome}

    resultado = await sessao.call_tool(nome, argumentos, read_timeout_seconds=PRAZO)
    return {
        "enviado": True,
        "ferramenta": nome,
        "resultado": resultado.content[0].text.replace("\n", " "),
    }


async def demonstrar_politica(sessao: ClientSession) -> None:
    """Executa três chamadas: permitida, sensível sem aprovação e fora da allowlist."""
    print_json(await chamar_com_politica(
        sessao, "consultar_cronograma", {"codigo_modulo": "A5"}))

    print_json(await chamar_com_politica(
        sessao, "resumir_modulo", {"codigo_modulo": "A5"}))

    print_json(await chamar_com_politica(
        sessao, "pastas_autorizadas", {}))


await com_sessao_curso(demonstrar_politica)

## 11.1 Discussão

As duas chamadas bloqueadas não chegaram ao transporte. Nenhuma mensagem JSON-RPC foi gerada, e o servidor não recebeu nenhuma das duas. É essa a diferença entre política no cliente e política no servidor: um servidor de terceiros não conhece as regras da aplicação que o consome, e recusar depois de executar não desfaz o efeito.

Os três controles cobrem alcances diferentes. `args`, na configuração da seção 8, limita o que o servidor enxerga do sistema. A allowlist limita o que o modelo alcança do catálogo. A aprovação limita o que acontece sem alguém olhando. O prazo limita quanto tempo uma chamada pode ocupar.

A lista de permitidas deste exemplo é fixa. Uma aplicação com vários servidores associa a política ao servidor de origem, porque a mesma ferramenta de escrita é aceitável em um diretório de trabalho e recusável em outro.

---

# 12. Exercícios de fixação

Os exercícios modificam exemplos já executados. Use as células vazias abaixo de cada enunciado.

## Exercício 1 — Descrição de um servidor

Escreva `async def descrever_servidor(sessao) -> dict`, que devolve o nome e a versão do servidor, as famílias de capacidade declaradas, a quantidade de ferramentas, resources e prompts, e os nomes das ferramentas que declaram `outputSchema`. Trate `McpError` em cada listagem e registre zero no lugar, porque um servidor que não declara a família recusa o método correspondente. Execute a função com `com_sessao_curso` e, depois, dentro de uma sessão com o servidor remoto da seção 7. Reporte em que campos os dois servidores diferem.

## Exercício 2 — Fronteira de privilégio

Crie o subdiretório `PASTA / "privado"` com um arquivo dentro. Acrescente a `CONFIG_MCP` uma quarta entrada, `filesystem_restrito`, apontando o servidor de arquivos apenas para `PASTA / "publico"`, que também precisa existir. Conecte as duas entradas de filesystem com `catalogo_do_host` e tente ler o arquivo de `privado` pelas duas. Reporte o que cada uma devolveu e qual campo da configuração produziu a diferença.

## Exercício 3 — Ferramenta nova no servidor

Acrescente ao arquivo `mcp_curso/servidor_curso.py` a ferramenta `listar_modulos()`, sem argumentos, anotada com o tipo de retorno `list[str]` e devolvendo os códigos dos três módulos. A alteração pode ser feita no arquivo ou reexecutando a célula da seção 1 com a função nova. Chame a ferramenta com `com_sessao_curso` e reporte quantos blocos a resposta trouxe em `content`. Depois acrescente o nome a `FERRAMENTAS_DO_AGENTE` e chame `responder("Quais módulos existem no curso?")`: o agente é construído dentro de `responder` e o catálogo é lido a cada sessão, então nenhuma outra célula precisa ser refeita.

## Exercício 4 — Cliente resiliente

Escreva `async def chamar_resiliente(sessao, nome, argumentos) -> dict` que trate os dois erros da seção 5: com `isError` verdadeiro, devolve `{"tipo": "execucao", "texto": ...}`; com `McpError`, devolve `{"tipo": "protocolo", "code": ..., "message": ...}`. Teste com `consultar_progresso` para o aluno `Pedro`, que não está registrado, e com `sessao.read_resource("curso://ementa")`, que não existe.

## Exercício 5 — Política por servidor

Generalize a política da seção 11 para o catálogo de vários servidores. Escreva `POLITICA = {"curso": "todas", "filesystem": {"read_text_file"}, "deepwiki": "nenhuma"}` e uma função que receba as sessões abertas por `catalogo_do_host` e devolva a lista de pares servidor e ferramenta que sobreviveram ao filtro. Reporte quantas ferramentas o modelo receberia com essa política e quantas existem no catálogo completo.

## Exercício 6 — Mensagens de uma leitura

Sem executar código, escreva a sequência de mensagens JSON-RPC que o traçado da seção 3 registraria para uma sessão que faz o handshake e depois lê o resource `curso://cronograma`. Indique, para cada mensagem, o sentido, se ela tem `id` e qual campo carrega o conteúdo da resposta.

---

# 13. Resumo da aula

O notebook consumiu servidores MCP a partir de um host, do handshake de uma conexão até um agente que escolhe ferramentas remotas com um modelo de linguagem. Em todos os arranjos, o cliente resolveu transporte, protocolo e descoberta, e as decisões de uso ficaram na aplicação.

| Conceito | Ideia principal |
|---|---|
| Host, cliente e servidor | O host mantém um cliente por servidor, com sessão e capacidades próprias. |
| `initialize` | Handshake que negocia a versão e informa quais famílias de método o servidor aceita. |
| JSON-RPC | Requisição com `id`, resposta com o mesmo `id`, notificação sem `id`. |
| Ferramenta, resource e prompt | Executar código, devolver conteúdo por URI e devolver mensagens parametrizadas. |
| `inputSchema` e `outputSchema` | Metadados que permitem usar um servidor desconhecido e validar o retorno. |
| `isError` e `McpError` | Falha na execução volta como resultado; falha de protocolo volta como exceção. |
| Transportes | Memória, stdio e HTTP mudam a conexão e não mudam as chamadas do cliente. |
| `mcpServers` | Arquivo de configuração onde cada entrada vira um cliente; `env` guarda segredos e `args` delimita privilégio. |
| Adaptação para LangChain | Nome, descrição e `inputSchema` de uma ferramenta MCP preenchem um `StructuredTool`. |
| `sampling` e `roots` | Requisições que o servidor faz ao cliente, atendidas por handlers registrados na sessão. |
| Política do cliente | Allowlist, prazo e aprovação aplicados antes de a requisição ser enviada. |

## 13.1 Checklist de compreensão

Questões de verificação:

1. Quais são as cinco tarefas de um cliente MCP, e quais decisões ficam fora dele?
2. O que o `initialize` devolve, e o que essa resposta autoriza?
3. Que campo liga uma resposta JSON-RPC ao pedido que a originou, e por que a ordem de chegada não basta?
4. Como um cliente descobre o modo de chamar uma ferramenta que nunca viu?
5. Qual a diferença de tratamento entre um resultado com `isError` verdadeiro e uma exceção `McpError`?
6. O que muda no código do cliente ao trocar stdio por HTTP, e o que permanece?
7. Quais três campos de uma ferramenta MCP são suficientes para construir uma ferramenta de LangChain?
8. Por que o `sampling` mantém no host o controle do modelo e do custo?
9. Por que a allowlist pertence ao cliente e não ao servidor?

## 13.2 Próximos passos

- Autenticação de servidores remotos privados por OAuth, com o fluxo de autorização dentro do host;
- `elicitation`, a terceira requisição do servidor ao cliente, para pedir um dado ao usuário durante a execução;
- escrita de servidores com efeito externo, e a interrupção do grafo para aprovação antes da execução;
- prefixo de origem no catálogo agregado, quando dois servidores declaram ferramentas de mesmo nome;
- registro e auditoria das chamadas MCP, com o servidor de origem, os argumentos e o resultado de cada uma;
- testes automatizados de um servidor próprio pelo transporte em memória.

---

# 14. Referências

- Model Context Protocol, arquitetura: https://modelcontextprotocol.io/specification/2025-11-25/architecture
- Model Context Protocol, ciclo de vida: https://modelcontextprotocol.io/specification/2025-11-25/basic/lifecycle
- Model Context Protocol, transportes: https://modelcontextprotocol.io/specification/2025-11-25/basic/transports
- Model Context Protocol, tools: https://modelcontextprotocol.io/specification/2025-11-25/server/tools
- Model Context Protocol, resources: https://modelcontextprotocol.io/specification/2025-11-25/server/resources
- Model Context Protocol, prompts: https://modelcontextprotocol.io/specification/2025-11-25/server/prompts
- Model Context Protocol, sampling: https://modelcontextprotocol.io/specification/2025-11-25/client/sampling
- Model Context Protocol, roots: https://modelcontextprotocol.io/specification/2025-11-25/client/roots
- SDK oficial em Python: https://github.com/modelcontextprotocol/python-sdk
- Servidores de referência: https://github.com/modelcontextprotocol/servers
- LangChain, ferramentas: https://docs.langchain.com/oss/python/langchain/tools
- Referência de API, `create_agent`: https://reference.langchain.com/python/langchain/agents/factory/create_agent
- Referência de API, `ChatOllama`: https://reference.langchain.com/python/langchain-ollama/chat_models/ChatOllama
- Referência de API, `ChatGoogleGenerativeAI`: https://reference.langchain.com/python/langchain-google-genai/chat_models/ChatGoogleGenerativeAI